# Fitts' Law Analysis - Mode Comparison Experiment

This notebook analyzes and compares Fitts' Law metrics (Throughput, Movement Time, etc.) between different participants and experimental conditions.

**Conditions:**
- 2 Modes: Rotation-Only vs 3-Point 2D
- 2 Variance Levels: Level 2 vs Level 3

## Step 1: Upload Your CSV Files

Run the cell below and upload your mode comparison CSV files.

In [ ]:
# Install and import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# Set style for better looking plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Libraries loaded successfully!")
print("\n" + "="*60)
print("Please upload your CSV files below:")
print("="*60)

In [ ]:
# Upload CSV files
print("Upload your CSV file (Student):")
uploaded_student = files.upload()
student_filename = list(uploaded_student.keys())[0]
print(f"\nUploaded: {student_filename}")

In [ ]:
# Upload Professor's CSV file
print("Upload Professor's CSV file:")
uploaded_prof = files.upload()
prof_filename = list(uploaded_prof.keys())[0]
print(f"\nUploaded: {prof_filename}")

## Step 2: Load and Process Data

In [ ]:
# Load the CSV files
df_student = pd.read_csv(student_filename)
df_professor = pd.read_csv(prof_filename)

# Add participant labels
df_student['participant'] = 'Student'
df_professor['participant'] = 'Professor'

# Combine datasets
df_all = pd.concat([df_student, df_professor], ignore_index=True)

# Create friendly mode names
df_all['mode_label'] = df_all['mode'].map({'rotation': 'Rotation-Only', 'threepoint': '3-Point 2D'})
df_all['variance_label'] = 'Variance ' + df_all['varianceLevel'].astype(str)
df_all['condition'] = df_all['mode_label'] + ' + ' + df_all['variance_label']

# Calculate throughput (IDe / MT)
df_all['throughput_calc'] = df_all['IDe'] / df_all['movementTime']

print(f"Student trials: {len(df_student)}")
print(f"Professor trials: {len(df_professor)}")
print(f"Total trials: {len(df_all)}")
print("\nData loaded successfully!")

## Step 3: Calculate Fitts' Law Metrics

In [ ]:
def calculate_fitts_metrics(df):
    """Calculate Fitts' Law metrics for each condition"""
    metrics = df.groupby(['participant', 'mode_label', 'varianceLevel']).agg({
        'movementTime': ['mean', 'std', 'count'],
        'IDe': ['mean', 'std'],
        'effectiveWidth': ['mean', 'std'],
        'selectionError': ['mean', 'std'],
        'throughput_calc': ['mean', 'std'],
        'actualAmplitude': ['mean']
    }).round(3)
    
    # Flatten column names
    metrics.columns = ['_'.join(col).strip() for col in metrics.columns.values]
    metrics = metrics.reset_index()
    
    return metrics

# Calculate metrics
metrics_df = calculate_fitts_metrics(df_all)

# Display summary table
print("="*80)
print("FITTS' LAW METRICS SUMMARY")
print("="*80)

# Create a cleaner summary table
summary = metrics_df[['participant', 'mode_label', 'varianceLevel', 
                       'movementTime_mean', 'movementTime_std',
                       'IDe_mean', 'effectiveWidth_mean', 
                       'selectionError_mean', 'throughput_calc_mean', 'throughput_calc_std']].copy()

summary.columns = ['Participant', 'Mode', 'Variance', 'MT (s)', 'MT SD', 
                   'IDe (bits)', 'We (px)', 'Error (px)', 'TP (bits/s)', 'TP SD']

print(summary.to_string(index=False))

## Step 4: Throughput Comparison Visualization

In [ ]:
# Create throughput comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Prepare data for plotting
pivot_tp = metrics_df.pivot_table(values='throughput_calc_mean', 
                                   index=['mode_label', 'varianceLevel'], 
                                   columns='participant')

# Plot 1: Grouped bar chart by condition
ax1 = axes[0]
x = np.arange(4)
width = 0.35

conditions = ['Rotation-Only\nVar 2', 'Rotation-Only\nVar 3', 
              '3-Point 2D\nVar 2', '3-Point 2D\nVar 3']

student_tp = metrics_df[metrics_df['participant'] == 'Student'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['throughput_calc_mean'].values
prof_tp = metrics_df[metrics_df['participant'] == 'Professor'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['throughput_calc_mean'].values

student_tp_std = metrics_df[metrics_df['participant'] == 'Student'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['throughput_calc_std'].values
prof_tp_std = metrics_df[metrics_df['participant'] == 'Professor'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['throughput_calc_std'].values

bars1 = ax1.bar(x - width/2, student_tp, width, label='Student', 
                yerr=student_tp_std, capsize=5, color='#2ecc71', alpha=0.8)
bars2 = ax1.bar(x + width/2, prof_tp, width, label='Professor', 
                yerr=prof_tp_std, capsize=5, color='#3498db', alpha=0.8)

ax1.set_ylabel('Throughput (bits/s)', fontsize=12)
ax1.set_title('Throughput Comparison by Condition', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(conditions)
ax1.legend()
ax1.set_ylim(0, max(max(student_tp), max(prof_tp)) * 1.3)

# Add value labels on bars
for bar, val in zip(bars1, student_tp):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, prof_tp):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 2: Heatmap comparison
ax2 = axes[1]
heatmap_data = metrics_df.pivot_table(values='throughput_calc_mean',
                                       index='participant',
                                       columns=['mode_label', 'varianceLevel'])
heatmap_data.columns = [f'{m}\nVar {v}' for m, v in heatmap_data.columns]

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
            ax=ax2, cbar_kws={'label': 'Throughput (bits/s)'},
            annot_kws={'size': 12, 'weight': 'bold'})
ax2.set_title('Throughput Heatmap', fontsize=14, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.savefig('throughput_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Figure saved as 'throughput_comparison.png'")

## Step 5: Movement Time Comparison

In [ ]:
# Movement Time comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Get MT data
student_mt = metrics_df[metrics_df['participant'] == 'Student'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['movementTime_mean'].values
prof_mt = metrics_df[metrics_df['participant'] == 'Professor'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['movementTime_mean'].values

student_mt_std = metrics_df[metrics_df['participant'] == 'Student'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['movementTime_std'].values
prof_mt_std = metrics_df[metrics_df['participant'] == 'Professor'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['movementTime_std'].values

# Bar chart
ax1 = axes[0]
bars1 = ax1.bar(x - width/2, student_mt, width, label='Student', 
                yerr=student_mt_std, capsize=5, color='#2ecc71', alpha=0.8)
bars2 = ax1.bar(x + width/2, prof_mt, width, label='Professor', 
                yerr=prof_mt_std, capsize=5, color='#3498db', alpha=0.8)

ax1.set_ylabel('Movement Time (seconds)', fontsize=12)
ax1.set_title('Movement Time Comparison by Condition', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(conditions)
ax1.legend()

# Add value labels
for bar, val in zip(bars1, student_mt):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
             f'{val:.1f}s', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, prof_mt):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
             f'{val:.1f}s', ha='center', va='bottom', fontsize=9)

# Box plot
ax2 = axes[1]
df_all['condition_short'] = df_all['mode_label'].str[:8] + '\n' + df_all['variance_label']
sns.boxplot(data=df_all, x='condition_short', y='movementTime', hue='participant', ax=ax2)
ax2.set_xlabel('')
ax2.set_ylabel('Movement Time (seconds)', fontsize=12)
ax2.set_title('Movement Time Distribution', fontsize=14, fontweight='bold')
ax2.legend(title='Participant')

plt.tight_layout()
plt.savefig('movement_time_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Figure saved as 'movement_time_comparison.png'")

## Step 6: Error Analysis

In [ ]:
# Selection Error comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Get error data
student_err = metrics_df[metrics_df['participant'] == 'Student'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['selectionError_mean'].values
prof_err = metrics_df[metrics_df['participant'] == 'Professor'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['selectionError_mean'].values

# Bar chart
ax1 = axes[0]
bars1 = ax1.bar(x - width/2, student_err, width, label='Student', color='#2ecc71', alpha=0.8)
bars2 = ax1.bar(x + width/2, prof_err, width, label='Professor', color='#3498db', alpha=0.8)

ax1.set_ylabel('Selection Error (pixels)', fontsize=12)
ax1.set_title('Selection Error Comparison', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(conditions)
ax1.legend()

for bar, val in zip(bars1, student_err):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val:.1f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, prof_err):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val:.1f}', ha='center', va='bottom', fontsize=9)

# Effective Width comparison
ax2 = axes[1]
student_we = metrics_df[metrics_df['participant'] == 'Student'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['effectiveWidth_mean'].values
prof_we = metrics_df[metrics_df['participant'] == 'Professor'].sort_values(
    ['mode_label', 'varianceLevel'], ascending=[False, True])['effectiveWidth_mean'].values

bars1 = ax2.bar(x - width/2, student_we, width, label='Student', color='#2ecc71', alpha=0.8)
bars2 = ax2.bar(x + width/2, prof_we, width, label='Professor', color='#3498db', alpha=0.8)

ax2.set_ylabel('Effective Width (pixels)', fontsize=12)
ax2.set_title('Effective Width (We) Comparison', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(conditions)
ax2.legend()

for bar, val in zip(bars1, student_we):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
             f'{val:.0f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, prof_we):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
             f'{val:.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('error_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Figure saved as 'error_comparison.png'")

## Step 7: Mode Comparison (Rotation vs 3-Point)

In [ ]:
# Compare modes across participants
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Throughput by Mode
ax1 = axes[0, 0]
mode_tp = df_all.groupby(['participant', 'mode_label'])['throughput_calc'].mean().unstack()
mode_tp.plot(kind='bar', ax=ax1, color=['#e74c3c', '#9b59b6'], alpha=0.8)
ax1.set_title('Throughput by Mode', fontsize=14, fontweight='bold')
ax1.set_ylabel('Throughput (bits/s)')
ax1.set_xlabel('')
ax1.legend(title='Mode')
ax1.set_xticklabels(['Student', 'Professor'], rotation=0)

# Movement Time by Mode
ax2 = axes[0, 1]
mode_mt = df_all.groupby(['participant', 'mode_label'])['movementTime'].mean().unstack()
mode_mt.plot(kind='bar', ax=ax2, color=['#e74c3c', '#9b59b6'], alpha=0.8)
ax2.set_title('Movement Time by Mode', fontsize=14, fontweight='bold')
ax2.set_ylabel('Movement Time (seconds)')
ax2.set_xlabel('')
ax2.legend(title='Mode')
ax2.set_xticklabels(['Student', 'Professor'], rotation=0)

# Throughput by Variance
ax3 = axes[1, 0]
var_tp = df_all.groupby(['participant', 'variance_label'])['throughput_calc'].mean().unstack()
var_tp.plot(kind='bar', ax=ax3, color=['#f39c12', '#1abc9c'], alpha=0.8)
ax3.set_title('Throughput by Variance Level', fontsize=14, fontweight='bold')
ax3.set_ylabel('Throughput (bits/s)')
ax3.set_xlabel('')
ax3.legend(title='Variance')
ax3.set_xticklabels(['Student', 'Professor'], rotation=0)

# Selection Error by Mode
ax4 = axes[1, 1]
mode_err = df_all.groupby(['participant', 'mode_label'])['selectionError'].mean().unstack()
mode_err.plot(kind='bar', ax=ax4, color=['#e74c3c', '#9b59b6'], alpha=0.8)
ax4.set_title('Selection Error by Mode', fontsize=14, fontweight='bold')
ax4.set_ylabel('Error (pixels)')
ax4.set_xlabel('')
ax4.legend(title='Mode')
ax4.set_xticklabels(['Student', 'Professor'], rotation=0)

plt.tight_layout()
plt.savefig('mode_variance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Figure saved as 'mode_variance_comparison.png'")

## Step 8: Summary Statistics Table

In [ ]:
# Create a nicely formatted summary table
print("\n" + "="*100)
print("FINAL SUMMARY: FITTS' LAW METRICS")
print("="*100)

# Pivot table for throughput
tp_pivot = metrics_df.pivot_table(
    values='throughput_calc_mean',
    index='participant',
    columns=['mode_label', 'varianceLevel'],
    aggfunc='first'
)

print("\n📊 THROUGHPUT (bits/second) - Higher is better")
print("-"*60)
print(tp_pivot.round(3).to_string())

# Pivot table for MT
mt_pivot = metrics_df.pivot_table(
    values='movementTime_mean',
    index='participant',
    columns=['mode_label', 'varianceLevel'],
    aggfunc='first'
)

print("\n\n⏱️ MOVEMENT TIME (seconds) - Lower is better")
print("-"*60)
print(mt_pivot.round(2).to_string())

# Pivot table for Error
err_pivot = metrics_df.pivot_table(
    values='selectionError_mean',
    index='participant',
    columns=['mode_label', 'varianceLevel'],
    aggfunc='first'
)

print("\n\n🎯 SELECTION ERROR (pixels) - Lower is better")
print("-"*60)
print(err_pivot.round(1).to_string())

# Overall comparison
print("\n\n" + "="*100)
print("OVERALL PARTICIPANT COMPARISON")
print("="*100)

overall = df_all.groupby('participant').agg({
    'throughput_calc': ['mean', 'std'],
    'movementTime': ['mean', 'std'],
    'selectionError': ['mean', 'std'],
    'IDe': 'mean'
}).round(3)

overall.columns = ['TP Mean', 'TP SD', 'MT Mean', 'MT SD', 'Error Mean', 'Error SD', 'IDe Mean']
print(overall.to_string())

## Step 9: Statistical Comparison

In [ ]:
from scipy import stats

print("\n" + "="*80)
print("STATISTICAL ANALYSIS")
print("="*80)

# T-tests between participants for each metric
print("\n📈 Independent t-tests (Student vs Professor):")
print("-"*60)

for metric, label in [('throughput_calc', 'Throughput'), 
                       ('movementTime', 'Movement Time'),
                       ('selectionError', 'Selection Error')]:
    student_data = df_all[df_all['participant'] == 'Student'][metric]
    prof_data = df_all[df_all['participant'] == 'Professor'][metric]
    
    t_stat, p_val = stats.ttest_ind(student_data, prof_data)
    
    sig = "*" if p_val < 0.05 else ""
    sig += "*" if p_val < 0.01 else ""
    sig += "*" if p_val < 0.001 else ""
    
    print(f"{label:20s}: t = {t_stat:7.3f}, p = {p_val:.4f} {sig}")

print("\n* p < 0.05, ** p < 0.01, *** p < 0.001")

# Mode comparison within each participant
print("\n\n📊 Mode Comparison (Rotation vs 3-Point) within each participant:")
print("-"*60)

for participant in ['Student', 'Professor']:
    rot_tp = df_all[(df_all['participant'] == participant) & 
                     (df_all['mode'] == 'rotation')]['throughput_calc']
    three_tp = df_all[(df_all['participant'] == participant) & 
                       (df_all['mode'] == 'threepoint')]['throughput_calc']
    
    t_stat, p_val = stats.ttest_ind(rot_tp, three_tp)
    
    print(f"\n{participant}:")
    print(f"  Rotation-Only TP: {rot_tp.mean():.3f} ± {rot_tp.std():.3f}")
    print(f"  3-Point 2D TP:    {three_tp.mean():.3f} ± {three_tp.std():.3f}")
    print(f"  Difference: t = {t_stat:.3f}, p = {p_val:.4f}")

## Step 10: Final Visualization - All Metrics

In [ ]:
# Create comprehensive summary figure
fig = plt.figure(figsize=(16, 10))

# Create grid
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# 1. Throughput comparison
ax1 = fig.add_subplot(gs[0, 0])
data_tp = metrics_df.pivot_table(values='throughput_calc_mean', 
                                  index=['mode_label', 'varianceLevel'],
                                  columns='participant')
data_tp.plot(kind='bar', ax=ax1, color=['#3498db', '#2ecc71'])
ax1.set_title('Throughput', fontweight='bold', fontsize=12)
ax1.set_ylabel('bits/s')
ax1.set_xlabel('')
ax1.legend(title='')
ax1.tick_params(axis='x', rotation=45)

# 2. Movement Time
ax2 = fig.add_subplot(gs[0, 1])
data_mt = metrics_df.pivot_table(values='movementTime_mean', 
                                  index=['mode_label', 'varianceLevel'],
                                  columns='participant')
data_mt.plot(kind='bar', ax=ax2, color=['#3498db', '#2ecc71'])
ax2.set_title('Movement Time', fontweight='bold', fontsize=12)
ax2.set_ylabel('seconds')
ax2.set_xlabel('')
ax2.legend(title='')
ax2.tick_params(axis='x', rotation=45)

# 3. Selection Error
ax3 = fig.add_subplot(gs[0, 2])
data_err = metrics_df.pivot_table(values='selectionError_mean', 
                                   index=['mode_label', 'varianceLevel'],
                                   columns='participant')
data_err.plot(kind='bar', ax=ax3, color=['#3498db', '#2ecc71'])
ax3.set_title('Selection Error', fontweight='bold', fontsize=12)
ax3.set_ylabel('pixels')
ax3.set_xlabel('')
ax3.legend(title='')
ax3.tick_params(axis='x', rotation=45)

# 4. Throughput distribution violin plot
ax4 = fig.add_subplot(gs[1, 0:2])
sns.violinplot(data=df_all, x='condition', y='throughput_calc', 
               hue='participant', split=True, ax=ax4)
ax4.set_title('Throughput Distribution by Condition', fontweight='bold', fontsize=12)
ax4.set_ylabel('Throughput (bits/s)')
ax4.set_xlabel('')
ax4.tick_params(axis='x', rotation=20)
ax4.legend(title='Participant', loc='upper right')

# 5. Summary metrics table as text
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')

summary_text = "SUMMARY\n" + "="*30 + "\n\n"
for participant in ['Student', 'Professor']:
    p_data = df_all[df_all['participant'] == participant]
    summary_text += f"{participant}:\n"
    summary_text += f"  Avg TP: {p_data['throughput_calc'].mean():.3f} bits/s\n"
    summary_text += f"  Avg MT: {p_data['movementTime'].mean():.2f} s\n"
    summary_text += f"  Avg Error: {p_data['selectionError'].mean():.1f} px\n\n"

ax5.text(0.1, 0.9, summary_text, transform=ax5.transAxes, fontsize=11,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Fitts\' Law Mode Comparison Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('comprehensive_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved as 'comprehensive_analysis.png'")

## Step 11: Download All Figures

In [ ]:
# Download all generated figures
from google.colab import files
import os

print("📥 Downloading figures...\n")

for filename in ['throughput_comparison.png', 'movement_time_comparison.png', 
                 'error_comparison.png', 'mode_variance_comparison.png',
                 'comprehensive_analysis.png']:
    if os.path.exists(filename):
        files.download(filename)
        print(f"  ✓ Downloaded: {filename}")

print("\n✅ All figures downloaded!")

## Step 12: Export Results to CSV

In [ ]:
# Export the summary metrics to CSV
summary_export = metrics_df[['participant', 'mode_label', 'varianceLevel', 
                              'movementTime_mean', 'movementTime_std', 'movementTime_count',
                              'IDe_mean', 'effectiveWidth_mean', 
                              'selectionError_mean', 'throughput_calc_mean', 'throughput_calc_std']].copy()

summary_export.columns = ['Participant', 'Mode', 'Variance Level', 
                          'MT Mean (s)', 'MT SD', 'Trial Count',
                          'IDe Mean (bits)', 'Effective Width (px)', 
                          'Error Mean (px)', 'Throughput Mean (bits/s)', 'Throughput SD']

summary_export.to_csv('fitts_analysis_results.csv', index=False)
files.download('fitts_analysis_results.csv')

print("\n✅ Results exported to 'fitts_analysis_results.csv'")
print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("="*60)